# Fine-tuning a Pretrained CNN on MNIST

Idea: take a CNN that's already pretrained (ResNet18 on ImageNet) and fine-tune it on MNIST digits instead of training from scratch. Should be way faster since the model already knows how to detect edges and shapes.

Plan: load resnet18, swap the last layer so it outputs 10 classes instead of 1000, train for a few epochs, save the weights so I can use them in the Gradio app later.

## 1. Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Load and prepare the MNIST dataset

MNIST images are grayscale (1 channel), but pretrained CNNs like ResNet expect 3-channel RGB images. We convert grayscale to 3 channels by repeating the single channel.

In [ ]:
# Transform: resize to 224x224 (ResNet input size), convert grayscale to 3 channels, normalize
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),  # repeat 1 channel into 3
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet stats
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

print(f'Train dataset: {len(train_dataset)} images')
print(f'Test  dataset: {len(test_dataset)} images')

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
# resize to 224x224 since that's what resnet expects, and repeat grayscale into 3 channels
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # imagenet normalization stats
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

print(f'train: {len(train_dataset)} images')
print(f'test: {len(test_dataset)} images')

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=2)


# checking the data loaded ok

In [ ]:
# Load pretrained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Replace the final classification layer: 512 -> 10 (digits 0-9)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)

model = model.to(device)
print(model.fc)  # confirm the new output layer
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

## Setting up the model

Loading resnet18 with pretrained weights, then replacing the last layer (model.fc) since the original one outputs 1000 classes for ImageNet and we only need 10.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# swap the last layer - originally 512 -> 1000, we want 512 -> 10
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)

model = model.to(device)
print(model.fc)
print(f'total params: {sum(p.numel() for p in model.parameters()):,}')


## loss + optimizer

using a small learning rate since we're fine-tuning, not training from scratch

In [ ]:
def compute_accuracy(outputs, labels):
    _, predicted = torch.max(outputs.data, 1)
    correct = (predicted == labels).sum().item()
    return correct / len(labels)

num_epochs = 3  # fine-tuning needs few epochs since the model is already pretrained

history = {'epoch': [], 'train_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(num_epochs):
    start = time.time()

    # --- Training ---
    model.train()
    running_loss = 0.0
    train_accs = []
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        train_accs.append(compute_accuracy(outputs, labels))

    # --- Validation ---
    model.eval()
    val_accs = []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_accs.append(compute_accuracy(outputs, labels))

    avg_loss = running_loss / len(train_loader)
    avg_train_acc = sum(train_accs) / len(train_accs)
    avg_val_acc = sum(val_accs) / len(val_accs)
    elapsed = time.time() - start

    history['epoch'].append(epoch + 1)
    history['train_loss'].append(avg_loss)
    history['train_acc'].append(avg_train_acc)
    history['val_acc'].append(avg_val_acc)

    print(f'Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f} | Train Acc: {avg_train_acc*100:.2f}% | Val Acc: {avg_val_acc*100:.2f}% | Time: {elapsed:.1f}s')

## Training loop

Only doing 3 epochs since the model is already pretrained, it doesn't need much more to adapt to MNIST. Tried 1 epoch first and accuracy was already pretty good so 3 should be plenty.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['epoch'], history['train_loss'], 'o-', color='tomato')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss'); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['epoch'], [a*100 for a in history['train_acc']], 'o-', color='steelblue', label='Train')
axes[1].plot(history['epoch'], [a*100 for a in history['val_acc']],   'o-', color='tomato',    label='Validation')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Train vs Validation Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## plotting loss + accuracy to see how it went

In [ ]:
torch.save(model.state_dict(), 'mnist_resnet18.pth')
print('Model weights saved as mnist_resnet18.pth')

import os
file_size = os.path.getsize('mnist_resnet18.pth') / (1024*1024)
print(f'File size: {file_size:.2f} MB')

## Saving the model

Saving just the state_dict (the weights) instead of the whole model, this is what's recommended, smaller file and works across different torch versions.

In [ ]:
# Create a fresh model with the same architecture
loaded_model = models.resnet18(weights=None)  # no pretrained weights this time
loaded_model.fc = nn.Linear(loaded_model.fc.in_features, 10)

# Load our fine-tuned weights into it
loaded_model.load_state_dict(torch.load('mnist_resnet18.pth', map_location=device))
loaded_model = loaded_model.to(device)
loaded_model.eval()

# Quick sanity check: predict on one test batch
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)
with torch.no_grad():
    outputs = loaded_model(images)
    _, predicted = torch.max(outputs, 1)

print('True labels:     ', labels[:10].cpu().numpy())
print('Predicted labels:', predicted[:10].cpu().numpy())
print(f'\nMatch: {(predicted[:10] == labels[:10]).sum().item()}/10 correct in this sample')

## double checking the saved file actually works

loading it back into a fresh model to make sure nothing got messed up when saving

In [ ]:
from google.colab import files
files.download('mnist_resnet18.pth')

## download the .pth file so I can upload it to Hugging Face later